# 01. Acquisition des graphes de réseau OSM

Téléchargement des réseaux piéton et cyclable via OSMnx sur le **périmètre
offre** (MRN + tampon de 5 000 m), à partir du polygone administratif généré
par `00_referentiels.ipynb`.

**Prérequis** : exécuter `00_referentiels.ipynb` au préalable  
**Entrée** : `data/perimetre_MRN.gpkg` périmètre administratif MRN  
**Source réseau** : OpenStreetMap via API Overpass  
**Projection** : EPSG:2154 (RGF93 / Lambert-93)  
**Sorties** :
- `data/reseau_pieton_MRN.gpkg` arêtes du réseau piéton (vecteur)
- `data/reseau_pieton_MRN.graphml` graphe piéton (topologie NetworkX)
- `data/reseau_velo_MRN.gpkg` arêtes du réseau cyclable (vecteur)
- `data/reseau_velo_MRN.graphml` graphe cyclable (topologie NetworkX)

> Les fichiers `.graphml` conservent la topologie du graphe nécessaire
> au calcul des isochrones (notebook 03). Les fichiers `.gpkg` sont
> destinés à la visualisation et au contrôle qualité dans QGIS.

In [ ]:
from pathlib import Path
import geopandas as gpd
import osmnx as ox

Détection de la racine du projet via le fichier sentinelle `.projectroot`,
puis ancrage de tous les chemins dessus. Cette approche est indépendante du
répertoire de travail : elle fonctionne sous JupyterLab comme sous VSCode,
quel que soit le réglage `notebookFileRoot`, et après un clone comme après un
téléchargement ZIP du dépôt.

In [ ]:
def trouver_racine(marqueur=".projectroot"):
    """Remonte depuis le cwd jusqu'au dossier contenant `marqueur`."""
    base = Path.cwd().resolve()
    for parent in (base, *base.parents):
        if (parent / marqueur).exists():
            return parent
    raise FileNotFoundError(f"Racine introuvable (marqueur {marqueur!r})")


ROOT = trouver_racine()
DATA_DIR = ROOT / "data"

## Acquisition sur le périmètre *offre*

Les graphes sont téléchargés non pas sur la MRN stricte mais sur un
**périmètre offre** = MRN + tampon de 5 000 m. Ce tampon évite que les
isochrones des arrêts proches de la limite (notebook 03) ne soient tronquées
faute de réseau au-delà du bord. Il est volontairement plus large que le
tampon de 3,75 km appliqué aux arrêts (notebook 02, = 15 min à vélo) :
router depuis un arrêt situé près du bord exige du réseau au-delà de cet arrêt.

Le tampon est appliqué en EPSG:2154 (en mètres), puis le polygone est
reprojeté en WGS84 (EPSG:4326), seul CRS accepté par `ox.graph_from_polygon`.
La population (périmètre *demande*) reste, elle, découpée à la MRN stricte
(notebook 04).

In [ ]:
ox.settings.timeout = 180
ox.settings.log_console = True
ox.settings.overpass_endpoint = "https://overpass-api.de/api"

# Périmètre offre = MRN + tampon, pour ne pas tronquer les isochrones des
# arrêts proches de la limite (cf. cadrage : offre vs demande). Le tampon est
# appliqué en CRS projeté (mètres), puis reprojeté en WGS84 pour OSMnx.
TAMPON_VIAIRE_M = 5000  # 5 km > 3,75 km (15 min vélo) : marge pour le routage

mrn = gpd.read_file(DATA_DIR / "perimetre_MRN.gpkg").to_crs("EPSG:2154")
zone_offre = mrn.geometry.union_all().buffer(TAMPON_VIAIRE_M)
polygon = gpd.GeoSeries([zone_offre], crs="EPSG:2154").to_crs("EPSG:4326").iloc[0]

G_walk = ox.graph_from_polygon(polygon, network_type="walk",
    retain_all=True)
G_walk_proj = ox.project_graph(G_walk, to_crs="EPSG:2154")
nodes_walk, edges_walk = ox.graph_to_gdfs(G_walk_proj)
edges_walk.to_file(DATA_DIR / "reseau_pieton_MRN.gpkg", driver="GPKG")
ox.save_graphml(G_walk_proj, filepath=DATA_DIR / "reseau_pieton_MRN.graphml")

G_bike = ox.graph_from_polygon(polygon, network_type="bike",
    retain_all=True)
G_bike_proj = ox.project_graph(G_bike, to_crs="EPSG:2154")
nodes_bike, edges_bike = ox.graph_to_gdfs(G_bike_proj)
edges_bike.to_file(DATA_DIR / "reseau_velo_MRN.gpkg", driver="GPKG")
ox.save_graphml(G_bike_proj, filepath=DATA_DIR / "reseau_velo_MRN.graphml")

print("Fichiers générés avec succès.")

In [ ]:

# Charger les graphes sauvegardés
G_walk = ox.load_graphml(DATA_DIR / "reseau_pieton_MRN.graphml")
G_bike = ox.load_graphml(DATA_DIR / "reseau_velo_MRN.graphml")

# Stats de base
for nom, G in [("Piéton", G_walk), ("Vélo", G_bike)]:
    stats = ox.basic_stats(G)
    print(f"\n── {nom} ──────────────────────")
    print(f"  Nœuds       : {G.number_of_nodes()}")
    print(f"  Arêtes      : {G.number_of_edges()}")
    print(f"  Longueur totale : {stats['edge_length_total']/1000:.1f} km")
    print(f"  Degré moyen : {stats['k_avg']:.2f}")
    print(f"  CRS         : {G.graph.get('crs', 'non défini')}")

In [ ]:
# Vérifier si le graphe est dirigé et compter les arêtes réciproques
print(f"Graphe piéton dirigé : {G_walk.is_directed()}")

# Compter les paires d'arêtes réciproques (A→B et B→A)
reciproques = sum(1 for u, v in G_walk.edges() if G_walk.has_edge(v, u))
print(f"Arêtes réciproques : {reciproques} / {G_walk.number_of_edges()}")

## Contrôle qualité des graphes

Rechargement des graphes depuis les fichiers `.graphml` et vérification
des indicateurs topologiques de base : nombre de nœuds et d'arêtes,
longueur totale du réseau, degré moyen et système de coordonnées.

| Indicateur        | Piéton     | Vélo       |
|-------------------|------------|------------|
| Nœuds             |  82 474    |  64 801    |
| Arêtes            | 222 326    | 153 035    |
| Longueur totale   |  20 014 km |  17 648 km |
| Degré moyen       | 5,39       | 4,72       |
| CRS               | EPSG:2154  | EPSG:2154  |

Le réseau piéton est plus dense que le cyclable, ce qui est cohérent :
OSMnx exclut du graphe vélo les voies sans accès cyclable déclaré dans OSM.

Le degré moyen élevé s'explique par la nature **dirigée** du graphe :
la totalité des arêtes (167 540/167 540) sont réciproques — chaque segment
est représenté dans les deux sens (A→B et B→A). Le degré effectif est
donc ~2,7 à pied et 2,4 à vélo, cohérent avec un réseau urbain mixte.
Sans impact sur les calculs d'isochrones.

La projection **EPSG:2154** est confirmée sur les deux graphes.